# Ember · 02 · Harmonize MODIS onto the VIIRS scale

Input: `hotspots_monthly.parquet` produced by notebook 01 (attach it as a notebook output source).
Outputs: `harmonized_monthly.parquet` (what the web app serves), `calibration_factors.parquet`, `validation.json` and the figures used in the pitch.

In [ ]:
MONTHLY = "/kaggle/input/ember-01-aggregate-firms/hotspots_monthly.parquet"   # TODO: adjust to the 01 output path
REPO = "https://github.com/GAUSS-TPAC/Ember---Harmonization-of-MODIS-and-VIIRS-Hot-Spots.git"

In [ ]:
import subprocess, sys
subprocess.run(["git", "clone", "--depth", "1", REPO, "/kaggle/working/ember"], check=True)
sys.path.insert(0, "/kaggle/working/ember/pipeline")
import json, time
import pandas as pd
import matplotlib.pyplot as plt
from ember_pipeline.harmonize import HarmonizationConfig, harmonize, validate

In [ ]:
monthly = pd.read_parquet(MONTHLY)
print(f"{len(monthly):,} rows, {monthly[['lat_cell','lon_cell']].drop_duplicates().shape[0]:,} cells")
monthly.groupby('platform')['n'].sum()

## The problem, in one figure: raw yearly totals per platform

In [ ]:
yearly = monthly.groupby(['platform', 'year'])['n'].sum().unstack(0)
ax = yearly.plot(figsize=(10, 4), marker='o')
ax.set_ylabel('raw detections / year'); ax.set_title('Before harmonization: the 2012 sensor jump')
plt.savefig('/kaggle/working/fig_raw_jump.png', dpi=150, bbox_inches='tight'); plt.show()

## Fit and apply the transfer factors

In [ ]:
cfg = HarmonizationConfig()
t0 = time.time()
harmonized, factors = harmonize(monthly, cfg)
print(f"{len(harmonized):,} harmonized rows in {time.time()-t0:.0f}s")
print(factors['level_MODIS'].value_counts())
factors[['k_MODIS', 'k_MODIS_Terra', 'k_VIIRS_NOAA20']].describe()

In [ ]:
harmonized.to_parquet('/kaggle/working/harmonized_monthly.parquet', index=False)
factors.to_parquet('/kaggle/working/calibration_factors.parquet', index=False)

## Validation on held-out years (fit 2012–2018, test 2019–2025)

In [ ]:
report = validate(monthly, fit_years=(2012, 2018), test_years=(2019, 2025), cfg=cfg)
json.dump(report, open('/kaggle/working/validation.json', 'w'), indent=2)
pd.DataFrame({'raw MODIS': report['raw'], 'harmonized': report['harmonized']})

## After: one continuous record

In [ ]:
series = harmonized.groupby(['year'])[['n_harm', 'n_low', 'n_high', 'n_modis', 'n_viirs_snpp']].sum()
fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(series.index, series['n_low'], series['n_high'], alpha=.25, label='uncertainty band')
ax.plot(series.index, series['n_harm'], marker='o', label='harmonized (VIIRS scale)')
ax.plot(series.index, series['n_modis'], ls='--', label='raw MODIS')
ax.set_title('After harmonization: yearly hotspots, world'); ax.legend()
plt.savefig('/kaggle/working/fig_harmonized.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Map of the transfer factor (how much more VIIRS sees than MODIS, per cell)
f = factors[factors['level_MODIS'] == 'cell']
fig, ax = plt.subplots(figsize=(12, 5))
sc = ax.scatter(f['lon_cell'], f['lat_cell'], c=f['k_MODIS'].clip(0, 12), s=1, cmap='viridis')
plt.colorbar(sc, label='k = VIIRS / MODIS'); ax.set_title('Transfer factor per 0.1° cell (overlap 2012–2025)')
plt.savefig('/kaggle/working/fig_factors_map.png', dpi=150, bbox_inches='tight'); plt.show()